In [ ]:
import time
import copy

import numpy as np

import matplotlib.pyplot as plt

import sys
sys.path.extend(['../../'])
from kmeans import Kmeans as KmeansTheory
from kmeans_plotting import Plot2DKmeans

# Clustering

Clustering refers to the set of techniques which allows to automatically group data into what are called clusters. There are many ways of implementing this process. From techniques that fix a maximum number of clusters, others that learn hierarchical clusters, and others that can learn the number of clusters automatically. 

Clustering is also grouped under a set of algorithms known as unsupervised learning, although I do not like this taxonomy. Also, many algorithms are not well known for being used as clustering algorithms (such as mixture models), but we can obtain clusters from their construction. 

Examples of algorithms that perform clustering include K-means, HDBSCAN, and Linear and Non-Linear Mixture Models and their infinite generalization through Dirichlet processes. There might be plenty more of them.

This assessment covers K-means.

## K-means: summary

K-means is an algorithm that assigns data to clusters based on computing distances, with a predefined number of clusters, which are then learnt automatically.

While it has a connection with Probabilistic Machine Learning (more precisely, it is a degenerate version of a Gaussian Mixture Model fitted through Maximum Log-Likelihood), the usual more intuitive design and description of the algorithm does not have a direct and clear connection with any of the usual mathematical probabilistic tools, in contrast with linear or logistic regression.

However, one can arrive at the algorithm and show that it is just the Expectation Maximization algorithm assuming a Gaussian mixture model where the variances are not learnt and set to equal values, but it is not so direct. Another way of arriving at the algorithm is by the optimization of a loss function, which, due to the optimization variables being involved (some of them being discrete), requires a coordinate descent optimization procedure. To my knowledge, this loss function does not have a probabilistic interpretation, in contrast to the approach that starts from a Gaussian Mixture Model. I would also need to check whether both approaches lead to the same optimization function, which I think it doesnt.

Since the loss function of the algorithm is non-convex, there are many local minima at which we can arrive. We will see that, in fact, initialization is a very important step in this algorithm.

The loss function targeted by this model is to minimize the within-cluster variance, defined by:

$$
\begin{split}
L(c,\mu,x) = \sum^K_{k=1}\sum_{n=1}^N r_{nk}\mid\mid x_n - \mu_k \mid\mid^2
\end{split}
$$

where $r_{nk}$ is an indicator variable such that:

$$
\begin{split}
r_{nk} = \begin{cases}
1, c_n = k\\
0, c_n \neq k
\end{cases}
\end{split}
$$

Note that here we have $K$ continous variables $\mu_k \in \mathbb{R}^d$ and $N$ discrete variables $c_n \in \mathbb{N}$.

Optimizing this loss function is done through coordinate descent, which results in the following algorithm:

* 1.  Initialization. Select the number of centroids to obtain and initialize them to any value. This could be done by selecting random points from the dataset, by selecting random points from the space of points, or by using advanced techniques such as the k-means++ algorithm.

After initialization, iterate the following steps:

* 2. Assign each data point to its closest centroid wrt some norm. The $L_2$ norm is usually used and is the one that connects with the Gaussian Mixture Model.
* 3. Recompute centroids by the sample mean of the data points assigned to each of the clusters.

Assume we have $N$ datapoints, with $K$ clusters $S=\{S_1,\dots,S_K\}$. Each point is denoted by $x_n \in \mathbb{R}^d$, each cluster centroid by $\mu_k$ and $c_n$ denotes the cluster being assigned to each $x_n$. Denote by $\mu = \{\mu_1,\mu_2,\dots,\mu_K\}$ and $c = \{c_1,c_2,\dots,c_N\}$.

Mathematically, we have:

* Step 2:

$$
\begin{split}
c_n = \underset{k}{\text{argmin}}\mid\mid x_n - \mu_k \mid\mid^2
\end{split}
$$

* Step 3: For each cluster $c$:

$$
\begin{split}
\mu_k = \frac{1}{\mid S_k \mid}\sum_{x_i \in S_k} x_i
\end{split}
$$

## Dataset

Create a dataset randomly to experiment with this assignment. Generate a total of $100$ sample points. You can sample these points randomly from some distribution, for example, a Gaussian. Another option is to sample it from a $3$ Gaussian mixture model to generate separate clusters.

Sampling from a mixture model can be done through ancestral sampling by first drawing a cluster randomly from the probability of the clusters $p(c)$ and then sampling from the conditional distributions of the data given the cluster $p(x\mid c)$. Since this is a Gaussian mixture model, we have $p(x\mid c)$ being a Gaussian distribution. Since the clusters are discrete random variables, we can use a categorical variable to model these clusters, with three categories, all of them being equally probable.

A good starting point is to assume:

$$
\begin{split}
p(c) &= \text{Cat}(c \mid p=(0.33, 0.33, 0.34))\\
p(x\mid c) &= \mathcal{N}(x \mid 0, 0.05 \cdot I )
\end{split}
$$


**Task 1:** Draw 100 points and show them in a 2-dimensional plot. This means that the Gaussian distributions being used are multivariate Gaussian distributions with $d=2$. Each of them will be parameterized with a mean vector and a covariance matrix. Use a diagonal covariance matrix with variance being $0.05$, which is given by $0, 0.05 \cdot I$.

Plot the data using matplotlib.

Remember to shuffle the data after its creation to improve the random initialization we will be doing later. This can be done through the following code:

```python
## shuffle data assuming X is a matrix where each row is a datapoint and columns represent data dimensions.
for i in range(10):
    np.random.shuffle(X)
```

To diagnose the follow-up implementation, seed the data generation using a seed of $1$, so that each time you run the cell, the exact same dataset is generated. This can be done through:

```
np.random.seed(1)
```

**Helper Code:**

In [ ]:
if False:
    np.random.seed(...)

    N_points = ...

    ## probability p(c)
    p = [...,...,...]
    c = [0,1,2]

    ## probability p(x|c)
    p_xc = {
        'c_0' : {
            'mu' : np.array([...,...]),
            'cov' : ...*np.eye(2),
            },
        'c_1' : {
            'mu' : np.array([...,...]),
            'cov' : ...*np.eye(2),
            },
        'c_2' : {
            'mu' : np.array([...,...]),
            'cov' : ...*np.eye(2),
            }
    }

    ## sample cluster assignments
    cluster = np.random.choice(c, size=N_points, p=p)

    ##
    X = np.zeros((N_points,2), dtype = np.float32)
    counter = 0
    X_clust = {}
    for _c in c:
        num_c = np.sum(cluster==_c)

        mu  = p_xc[f'c_{_c}']['mu']
        cov = p_xc[f'c_{_c}']['cov']

        _x = ...
        X[counter:counter + num_c] = _x

        X_clust[_c] = _x

        counter += num_c

    ## display
    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
    ax1.plot(...)
    ax1.set_xlabel(...)
    ax1.set_ylabel(...)
    ax1.set_title(...)

    for _c in c:
        _x = X_clust[_c]
        ax2.plot(...)
        ax2.set_xlabel(...)
        ax2.set_ylabel(...)
        ax2.set_title(...)


    ## shuffle data
    for i in range(10):
        np.random.shuffle(X)

**Solution:**

In [ ]:
np.random.seed(1)

N_points = 100

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
p_xc = {
    'c_0' : {
        'mu' : np.array([0,0]),
        'cov' : 0.05*np.eye(2),
        },
    'c_1' : {
        'mu' : np.array([0.5,0.5]),
        'cov' : 0.05*np.eye(2),
        },
    'c_2' : {
        'mu' : np.array([0,0.5]),
        'cov' : 0.05*np.eye(2),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,2), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    X[counter:counter + num_c] = _x
    
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],X[:,1], 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],_x[:,1], 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_ylabel(r"$x_2$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

## Algorithm

Implement the K-means algorithm. For this task, we will use a different programming approach than for linear regression and will use classes instead of functional programming.

**Task 2:**  Implement the class method of kmeans. Start by implementing them in the order being used. This means you should start with `initialize_centroids`, then follow with `_assign_data_to_centroids`, then `_get_new_centroids`, and end with `run_iter` and `run`. To initialize centroids, you could use `np.random.randint` to select randomly from within the dataset.

**Helper Code:**

In [ ]:
if False:
    class Kmeans:
        def __init__(self, num_centroids : int):
            self.num_centroids = ...

        def _assign_data_to_centroids(self, X):
            """
            Assigns data X to centroids. Step 2 of the algorithm
            """

            ## compute distance to centroids
            dist = ...

            ## assign the centroid with lowest distance
            centroids_assigned = ...

            ## separate data X by assigned centroid
            X_assigned = {}

            for _c in range(self.num_centroids):
                _x_assign = ...
                X_assigned[_c] = ...

            return centroids_assigned, X_assigned

        def _get_new_centroids(self,X):
            '''
            Compute new centroids given assignments, step 3 in the algorithm.
            '''
            ## get X assigned to centroids
            ...

            ## for each assignment compute new centroid
            for _c in range(self.num_centroids):
                self._centroids[_c] = ...

            return centroids_assigned, X_assigned

        def run_iter(self,X):
            ''' Run a single iteration of the algorithm '''
            return ...

        def run(self,X, num_iters, seed = None):
            '''Run num iter times the algorithm

            The seed is used to fix initialization
            '''

            ## initialize centroids
            ...

            for itet in range(num_iters):
                print(f"Running iteration {itet}", end = "\r")

                centroids_assigned, X_assigned = self.run_iter(X)

                # will give values to loss later
                loss = 0.0
                print(f"Running iteration {itet} loss {loss:.3f}")

                ## if current centroids are the same as previous stop
                if np.all(self.centroids == centroids_old):
                    break

        def initialize_centroids(self,X, seed = None):

            if seed is not None:
                np.random.seed(seed)

            # we start by chosing randomly from X as many centroids as we want.
            centroids_idx = ...

            self._centroids = X[centroids_idx]

            return self._centroids

**Solution:**

In [ ]:
class Kmeans:
    def __init__(self, num_centroids : int):
        self.num_centroids = num_centroids

    @property
    def centroids(self):
        return copy.deepcopy(self._centroids)

    def _assign_data_to_centroids(self,X, centroids = None):
        """
        Assigns data X to centroids.
        """

        if centroids is None:
            centroids = self._centroids

        ## compute distance to centroids
        dist = np.sum((X[:,np.newaxis] - centroids)**2,axis=-1)

        ## assign the centroid with lowest distance
        centroids_assigned = np.argmin(dist, axis = -1)

        ## separate data X by assigned centroid
        X_assigned = {}

        for _c in range(self.num_centroids):
            _x_assign = X[centroids_assigned == _c]
            X_assigned[_c] = _x_assign

        return centroids_assigned, X_assigned

    def _get_new_centroids(self,X):
        centroids_assigned, X_assigned = self._assign_data_to_centroids(X)

        for _c in range(self.num_centroids):
            _x_assign = X_assigned[_c]
            # if no data then keep old centroid
            if _x_assign.size != 0:
                self._centroids[_c] = np.mean(X_assigned[_c], axis = 0)

        return centroids_assigned, X_assigned

    def run_iter(self,X):
        return self._get_new_centroids(X)

    def run(self,X, num_iters, seed = None):

        self.initialize_centroids(X, seed = seed)

        for itet in range(num_iters):
            print(f"Running iteration {itet}", end = "\r")
            centroids_old = copy.deepcopy(self.centroids)

            centroids_assigned, X_assigned = self.run_iter(X)

            loss = self.loss_function(X, X_assigned = X_assigned)

            print(f"Running iteration {itet} loss {loss:.3f}")

            if np.all(self.centroids == centroids_old):
                break

    def initialize_centroids(self,X, seed = None):
        if seed is not None:
            np.random.seed(seed)

        # we start by chosing randomly from X.
        centroids_idx = np.random.randint(0, X.shape[0], size=self.num_centroids)

        self._centroids = X[centroids_idx]

        return self._centroids

    def loss_function(self, X, centroids = None, X_assigned = None):

        if centroids is None:
            centroids = self.centroids
            num_centroids = self.num_centroids
        else:
            num_centroids = centroids.shape[0]

        if X_assigned is None:
            _, X_assigned = self._assign_data_to_centroids(X, centroids = centroids)

        loss = 0.0
        for _c in range(num_centroids):
            loss += np.sum((centroids[_c] - X_assigned[_c])**2)

        return loss

**Task 3:** Run the algorithm for 20 epochs, passing a seed of 1 so that centroids get initialized by that seed, the final centroids to obtain should be the following:

**Helper Code:**

In [ ]:
if False:
    kmeans = Kmeans(num_centroids = ...)
    kmeans.run(X, num_iters = ..., seed = ...)

    print("==================")
    print("Centroids Obtained")
    print("==================")
    print(kmeans.centroids)

**Solution:**

In [ ]:
kmeans = Kmeans(num_centroids = 4)
kmeans.run(X, num_iters = 20, seed = 1)

print("==================")
print("Centroids Obtained")
print("==================")
print(kmeans.centroids)

## Loss function implementation

**Task 4:** Add a new method to the Kmeans class that implements the loss function, with the following signature (it is already included in the complete class defined above, but let's isolate it as its own exercise):

Now, in the run method, call the loss method just after calling the run_iter method to display the loss over the course of learning, after each iteration. After that, run the code, and you should get the same losses as in the cell above.

**Helper Code:**

In [ ]:
if False:
    ## Add method to your Kmeans class creation.
    def loss_function(self, X, centroids = None, X_assigned = None):
        ''' Compute the loss function'''

        if centroids is None:
            centroids = self.centroids
            num_centroids = self.num_centroids
        else:
            num_centroids = centroids.shape[0]

        if X_assigned is None:
            ...

        loss = 0.0
        for _c in range(num_centroids):
            loss += ...

        return loss

**Solution:**

The function is already included in the solution to task 2.

## Plotting

Now let's try to visualize the learning algorithm. The way plotting is done in the theory requires a bit of programming complexity to create code that is easily adjustable to the different sections with minimum modifications. However, for the didactic purposes of this assessment, this should not be necessary at all.

**Task 5:** Implement the following functions to visualize the algorithm:

1. `draw_initial_centroids`: draw current centroids.
2. `draw_cluster_assignments`: draw data assignments to clusters.
3. `draw_voronoi`: draw Voronoi regions.
4. `draw_new_centroids`: draw a new centroid.
5. `draw_loss`: draw the current loss value.

They will be called in the following order: `draw_initial_centroids` is called once, right after the centroids are initialized. Then, on every iteration of the algorithm, in this order: `draw_cluster_assignments` (using the centroids and their assignments *before* the update), `draw_new_centroids` (using the updated centroids), and finally `draw_loss`. `draw_voronoi` is never called directly by this sequence — it is used internally by the other three functions to draw the decision boundary on top of whatever they are already plotting.

Do it in your favourite way. Try to think about how to do it, experiment with it, and try to optimize your code to get the cleanest implementation. A good tip when programming is that methods, functions, etc should do very precise tasks. This means that, rather than modifying a method that performs some computation (for example, centroid assignment) to display (the centroid assignments), it is preferable to implement a method that exclusively plots (centroid assignments).

Voronoi thresholds can be obtained by defining a 2-dimensional grid using meshgrid, computing on each point of the grid which cluster would be assigned, and then displaying that mesh using contour from matplotlib.

You can use `matplotlib tk` in a Jupyter cell to get figure updates. Basically, use the same code you used in regression assessment to create interactive plots that get updated on each display. Remember, things can be done with (when using Jupyter):

```python
%matplotlib tk
ax.cla()
...
fig.canvas.draw()
fig.canvas.flush_events()
time.sleep(sleep_time)
```

**Helper Code:**

In [ ]:
if False:
    def draw_voronoi(ax, centroids, colors = 'k'):
        '''
        Draw the Voronoi regions (cluster decision boundaries) for the given centroids.

        ax: matplotlib axis to draw on.
        centroids: array of shape (num_centroids, 2) with the centroid positions.
        colors: color used to draw the region boundaries.
        '''
        xx, yy = np.meshgrid(...)
        voronoi_regions, _ = assign_data_to_centroids(np.c_[xx.ravel(), yy.ravel()], centroids)

        voronoi_regions = np.reshape(voronoi_regions, xx.shape)
        ax.contour(...)

    def draw_initial_centroids(ax, centroids, X, sleep_time = 0.3):
        '''
        Draw the dataset and the initial centroids, together with their Voronoi regions.

        ax: matplotlib axis to draw on.
        centroids: array of shape (num_centroids, 2) with the initial centroid positions.
        X: array of shape (N, 2) with the dataset.
        sleep_time: seconds to pause after drawing, to allow the figure to be seen when updated live.
        '''
        ax.cla()
        ax.plot(...)

        for idx,cet in enumerate(centroids):
            ax.plot(...)

        ax.set_title(...)
        draw_voronoi(...)
        ax.legend()

        ax.figure.canvas.draw()
        ax.figure.canvas.flush_events()
        time.sleep(sleep_time)

    def draw_cluster_assignments(ax, centroids, X_assigned, sleep_time = 0.3):
        '''
        Draw the centroids together with the data assigned to each of them, and their Voronoi regions.

        ax: matplotlib axis to draw on.
        centroids: array of shape (num_centroids, 2) with the centroid positions used for the assignment.
        X_assigned: dict mapping each centroid index to the array of points assigned to it.
        sleep_time: seconds to pause after drawing, to allow the figure to be seen when updated live.
        '''
        ax.cla()

        for idx,cet in enumerate(centroids):
            label = ""
            if idx == 0:
                label = "Centroid"

            ax.plot(...)

        for _c in range(centroids.shape[0]):
            _X = X_assigned[_c]
            ax.plot(...)

        ax.set_title(...)
        draw_voronoi(...)
        ax.legend()

        ax.figure.canvas.draw()
        ax.figure.canvas.flush_events()
        time.sleep(sleep_time)

    def draw_new_centroids(ax, centroids, sleep_time = 0.3):
        '''
        Draw the updated centroids (after recomputation), together with their Voronoi regions.

        ax: matplotlib axis to draw on.
        centroids: array of shape (num_centroids, 2) with the updated centroid positions.
        sleep_time: seconds to pause after drawing, to allow the figure to be seen when updated live.
        '''
        for idx,cet in enumerate(centroids):
            label = ""
            if idx == 0:
                label = "New centroid"

            ax.plot(...)

        ax.set_title(...)
        draw_voronoi(...)
        ax.legend()

        ax.figure.canvas.draw()
        ax.figure.canvas.flush_events()
        time.sleep(sleep_time)

    def draw_loss(ax, loss, sleep_time = 0.3):
        '''
        Display the current loss value, e.g. as the plot title.

        ax: matplotlib axis to draw on.
        loss: scalar with the current loss value.
        sleep_time: seconds to pause after drawing, to allow the figure to be seen when updated live.
        '''
        ax.set_title(...)

        ax.figure.canvas.draw()
        ax.figure.canvas.flush_events()
        time.sleep(sleep_time)

**Solution:**

In [ ]:
def draw_voronoi(ax, centroids, colors = 'k'):
    xx, yy = np.meshgrid(np.linspace(-1, 1, 500), np.linspace(-1, 1, 500))
    voronoi_regions, _ = assign_data_to_centroids(np.c_[xx.ravel(), yy.ravel()], centroids)
    voronoi_regions = np.reshape(voronoi_regions, xx.shape)
    ax.contour(xx, yy, voronoi_regions, colors = colors, linewidths = 1)

def draw_initial_centroids(ax, centroids, X, sleep_time = 0.3):
    ax.cla()
    ax.plot(X[:,0], X[:,1], 'x', color = 'k')

    for idx,cet in enumerate(centroids):
        ax.plot(cet[0], cet[1], 'o', color = f'C{idx}')

    ax.set_title("Initial Centroids")
    draw_voronoi(ax, centroids)
    ax.legend()

    ax.figure.canvas.draw()
    ax.figure.canvas.flush_events()
    time.sleep(sleep_time)

def draw_cluster_assignments(ax, centroids, X_assigned, sleep_time = 0.3):
    ax.cla()

    for idx,cet in enumerate(centroids):
        label = ""
        if idx == 0:
            label = "Centroid"

        ax.plot(cet[0], cet[1], 'o', color = f'C{idx}', label = label)

    for _c in range(centroids.shape[0]):
        _X = X_assigned[_c]
        ax.plot(_X[:,0], _X[:,1], 'x', color = f'C{_c}')

    ax.set_title("Centroids Assignment")
    draw_voronoi(ax, centroids)
    ax.legend()

    ax.figure.canvas.draw()
    ax.figure.canvas.flush_events()
    time.sleep(sleep_time)

def draw_new_centroids(ax, centroids, sleep_time = 0.3):
    for idx,cet in enumerate(centroids):
        label = ""
        if idx == 0:
            label = "New centroid"

        ax.plot(cet[0], cet[1], '*', color = f'C{idx}', markersize = 20, label = label)

    ax.set_title("New Centroids")
    draw_voronoi(ax, centroids, colors = 'gray')
    ax.legend()

    ax.figure.canvas.draw()
    ax.figure.canvas.flush_events()
    time.sleep(sleep_time)

def draw_loss(ax, loss, sleep_time = 0.3):
    ax.set_title(f"Loss {loss :.3f}")

    ax.figure.canvas.draw()
    ax.figure.canvas.flush_events()
    time.sleep(sleep_time)

**Task 6:** Now let's connect the plotting functions from the previous task with the actual algorithm. Add a new argument `interactive_plot` (defaulting to `False`) to the `__init__` method of your `Kmeans` class. When `interactive_plot=True`, create a matplotlib figure and axis (e.g. with `plt.subplots()`) and store them as attributes of the instance (e.g. `self.fig`, `self.ax`).

Then, modify `initialize_centroids` and `run` so that, whenever `self.interactive_plot` is `True`, they call the drawing functions you implemented in the previous task, passing `self.ax`, in the exact order already described: `draw_initial_centroids` once, right after the centroids are initialized, and then, on every iteration, `draw_cluster_assignments`, `draw_new_centroids` and `draw_loss`, in that order.

**Helper Code:**

In [ ]:
if False:
    class Kmeans:
        def __init__(self, num_centroids : int, interactive_plot = False):
            self.num_centroids = num_centroids

            self.interactive_plot = interactive_plot

            if self.interactive_plot:
                self.fig, self.ax = ...

        def run(self,X, num_iters, seed = None):

            self.initialize_centroids(X, seed = seed)

            for itet in range(num_iters):
                print(f"Running iteration {itet}", end = "\r")
                centroids_old = copy.deepcopy(self.centroids)

                centroids_assigned, X_assigned = self.run_iter(X)

                loss = self.loss_function(X, X_assigned = X_assigned)

                ## plot
                if self.interactive_plot:
                    ...

                print(f"Running iteration {itet} loss {loss:.3f}")

                if np.all(self.centroids == centroids_old):
                    break

        def initialize_centroids(self,X, seed = None):
            if seed is not None:
                np.random.seed(seed)

            # we start by chosing randomly from X.
            centroids_idx = np.random.randint(0, X.shape[0], size=self.num_centroids)

            self._centroids = X[centroids_idx]

            if self.interactive_plot:
                ...

            return self._centroids

**Solution:**

In [ ]:
class Kmeans:
    def __init__(self, num_centroids : int, interactive_plot = False):
        self.num_centroids = num_centroids

        self.interactive_plot = interactive_plot

        if self.interactive_plot:
            self.fig, self.ax = plt.subplots()

    @property
    def centroids(self):
        return copy.deepcopy(self._centroids)

    def _assign_data_to_centroids(self,X, centroids = None):
        """
        Assigns data X to centroids.
        """

        if centroids is None:
            centroids = self._centroids

        ## compute distance to centroids
        dist = np.sum((X[:,np.newaxis] - centroids)**2,axis=-1)

        ## assign the centroid with lowest distance
        centroids_assigned = np.argmin(dist, axis = -1)

        ## separate data X by assigned centroid
        X_assigned = {}

        for _c in range(self.num_centroids):
            _x_assign = X[centroids_assigned == _c]
            X_assigned[_c] = _x_assign

        return centroids_assigned, X_assigned

    def _get_new_centroids(self,X):
        centroids_assigned, X_assigned = self._assign_data_to_centroids(X)

        for _c in range(self.num_centroids):
            _x_assign = X_assigned[_c]
            # if no data then keep old centroid
            if _x_assign.size != 0:
                self._centroids[_c] = np.mean(X_assigned[_c], axis = 0)

        return centroids_assigned, X_assigned

    def run_iter(self,X):
        return self._get_new_centroids(X)

    def run(self,X, num_iters, seed = None):

        self.initialize_centroids(X, seed = seed)

        for itet in range(num_iters):
            print(f"Running iteration {itet}", end = "\r")
            centroids_old = copy.deepcopy(self.centroids)

            centroids_assigned, X_assigned = self.run_iter(X)

            loss = self.loss_function(X, X_assigned = X_assigned)

            ## plot
            if self.interactive_plot:
                draw_cluster_assignments(self.ax, centroids_old, X_assigned)
                draw_new_centroids(self.ax, self._centroids)
                draw_loss(self.ax, loss)

            print(f"Running iteration {itet} loss {loss:.3f}")

            if np.all(self.centroids == centroids_old):
                break

    def initialize_centroids(self,X, seed = None):
        if seed is not None:
            np.random.seed(seed)

        # we start by chosing randomly from X.
        centroids_idx = np.random.randint(0, X.shape[0], size=self.num_centroids)

        self._centroids = X[centroids_idx]

        if self.interactive_plot:
            draw_initial_centroids(self.ax, self._centroids, X)

        return self._centroids

    def loss_function(self, X, centroids = None, X_assigned = None):

        if centroids is None:
            centroids = self.centroids
            num_centroids = self.num_centroids
        else:
            num_centroids = centroids.shape[0]

        if X_assigned is None:
            _, X_assigned = self._assign_data_to_centroids(X, centroids = centroids)

        loss = 0.0
        for _c in range(num_centroids):
            loss += np.sum((centroids[_c] - X_assigned[_c])**2)

        return loss

**Task 7:** Now run the algorithm with `interactive_plot=True` and visualize the solution. Remember to switch to an interactive matplotlib backend (`%matplotlib tk`) beforehand if you want to see the plot update live as the algorithm runs; with the default inline backend (`%matplotlib inline`) you will only see the final frame.

**Helper Code:**

In [ ]:
if False:
    interactive_plot = ...

    if interactive_plot:
        %matplotlib tk
        plt.close("all")
    else:
        %matplotlib inline

    kmeans = Kmeans(num_centroids = ..., interactive_plot = ...)
    kmeans.run(X, num_iters = ..., seed = ...)
    %matplotlib inline

**Solution:**

In [ ]:
if False:
    interactive_plot = True

    if interactive_plot:
        %matplotlib tk
        plt.close("all")
    else:
        %matplotlib inline

    kmeans = Kmeans(num_centroids = 4, interactive_plot = interactive_plot)
    kmeans.run(X, num_iters = 20, seed = 1)
    %matplotlib inline

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline

plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = False
                      )

kmeans = KmeansTheory(num_centroids = 4, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 1)

plotter.show_video()

## Loss function plotting

For plotting the loss function, we will just focus on a simple example on 1-dimensional data. Interested readers can inspect the code from the theory to see how to plot two-dimensional data.

**Task 8:** Create a 1-dimensional dataset following the same recipe as before, this time with $3$ points and three components of different variance ($0.1$, $0.01$ and $0.01$) centered at $0$, $3$ and $5$.

**Helper Code:**

In [ ]:
if False:
    np.random.seed(...)

    N_points = ...

    ## probability p(c)
    p = [...,...,...]
    c = [0,1,2]

    ## probability p(x|c)
    var1 = ...
    var2 = ...
    var3 = ...
    p_xc = {
        'c_0' : {
            'mu' : np.array([...]),
            'cov' : var1*np.eye(1),
            },
        'c_1' : {
            'mu' : np.array([...]),
            'cov' : var2*np.eye(1),
            },
        'c_2' : {
            'mu' : np.array([...]),
            'cov' : var3*np.eye(1),
            }
    }

    ## sample cluster assignments
    cluster = np.random.choice(c, size=N_points, p=p)

    ##
    X = np.zeros((N_points,1), dtype = np.float32)
    counter = 0
    X_clust = {}
    for _c in c:
        num_c = np.sum(cluster==_c)

        mu  = p_xc[f'c_{_c}']['mu']
        cov = p_xc[f'c_{_c}']['cov']

        _x = ...

        X[counter:counter + num_c] = _x
        X_clust[_c] = _x

        counter += num_c

    fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
    ax1.plot(...)
    ax1.set_xlabel(...)
    ax1.set_title(...)

    for _c in c:
        _x = X_clust[_c]
        ax2.plot(...)
        ax2.set_xlabel(...)
        ax2.set_title(...)


    ## shuffle data
    for i in range(10):
        np.random.shuffle(X)

**Solution:**

In [ ]:
np.random.seed(1)

N_points = 3

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([3]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([5]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

**Task 9:** Get two initial centroids. You can do it by selecting randomly from these 3 points, or just get you kmeans implementation and call the method initialize_centroids. You can use as many centroids as you want. Start with two, and then we will see other options.

**Helper Code:**

In [ ]:
if False:
    kmeans = Kmeans(num_centroids = ...)
    kmeans.initialize_centroids(X, seed = ...)

    print("==================")
    print("Selected centroids")
    print(kmeans.centroids)

**Solution:**

In [ ]:
kmeans = Kmeans(num_centroids = 2)
kmeans.initialize_centroids(X, seed = 1)

print("==================")
print("Selected centroids")
print(kmeans.centroids)

**Task 10:** Draw the loss function by fixing one centroid to a value and iterating over other values for the other centroid; compute the loss and display it. Do not focus on displaying colors within the different regions. Once you have the code, try to do it. The idea is to assign a color to each region, defined by having the same cluster assignments.

**Helper Code:**

In [ ]:
if False:
    # Plotting specifications
    N_grid = ...
    grid_lim_l = ...
    grid_lim_u = ...

    ## coordinate range of values to change
    c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

    fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,1, figsize = (10,5))

    ## ====================
    ## Parameter to vary ##
    ## Centroid and coordinate

    ## to separate the cost function within the different assignments
    assigned_regions = {}

    # to keep loss
    loss_acc = {}

    # to keep region colors
    region_colors = {}

    for cent2change in range(kmeans.num_centroids):
        
        ## to separate the cost function within the different assignments
        assigned_regions[cent2change] = []

        # Get initial centroids
        centroids = copy.deepcopy(kmeans.centroids)

        # to keep loss
        loss_acc[cent2change] = []
        
        for _c in c_coord_range:
            centroids[cent2change,:] = _c

            centroids_assigned, X_assigned = ...

            assigned_regions[cent2change].append(tuple(centroids_assigned))

            loss = ...
            loss_acc[cent2change].append(loss)
     
        unique_assignments = list(set(assigned_regions[cent2change]))
        region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
       
        ## ==================
        ## Draw loss function
        ax_list_1[cent2change].plot(...)
        ax_list_1[cent2change].scatter(...)

        ax_list_1[cent2change].set_xlabel(...)

**Solution:**

In [ ]:
# Plotting specifications
N_grid = 40
grid_lim_l = -10
grid_lim_u = 10

## coordinate range of values to change
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,1, figsize = (12,8))

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    # Get initial centroids
    centroids = copy.deepcopy(kmeans.centroids)

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)
 
    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change].set_xlabel(f"Centroid {cent2change}")
    

**Task 11:** Draw loss function changing the number of points created to 8.

**Helper code:**

You just need to repeat the three steps above but creating 8 points.

**Solution:**

In [ ]:
np.random.seed(1)

N_points = 8

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([3]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([5]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
kmeans = Kmeans(num_centroids = 2)
kmeans.initialize_centroids(X, seed = 1)

print("==================")
print("Selected centroids")
print(kmeans.centroids)

In [ ]:
# Plotting specifications
N_grid = 40
grid_lim_l = -10
grid_lim_u = 10

## coordinate range of values to change
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,1, figsize = (12,8))

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    # Get initial centroids
    centroids = copy.deepcopy(kmeans.centroids)

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)
 
    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change].set_xlabel(f"Centroid {cent2change}")
    

**Task 12:** Now vary the number of centroids and see different loss functions.

**Solution:**

In [ ]:
kmeans = Kmeans(num_centroids = 5)
kmeans.initialize_centroids(X, seed = 1)

print("==================")
print("Selected centroids")
print(kmeans.centroids)

In [ ]:
# Plotting specifications
N_grid = 100
grid_lim_l = -10
grid_lim_u = 10
## coordinate range of values to change
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,1, figsize = (10,20))

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    # Get initial centroids
    centroids = copy.deepcopy(kmeans.centroids)

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)
 
    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change].set_xlabel(f"Centroid {cent2change}")

**Task 13:** Given a cluster assignment, all the loss functions are convex. Modify your previous code so that the loss function is always computed with the same assignment to see this convexity.

**Helper Code:**

In [ ]:
if False:
    # Plotting specifications
    N_grid = ...
    grid_lim_l = ...
    grid_lim_u = ...

    ## coordinate range of values to change
    c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

    fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,1, figsize = (10,20))

    ## always use the same (fixed) assignment, computed once here
    true_centroids_assigned, true_X_assigned = ...

    ## ====================
    ## Parameter to vary ##
    ## Centroid and coordinate

    ## to separate the cost function within the different assignments
    assigned_regions = {}

    # to keep loss
    loss_acc = {}

    # to keep region colors
    region_colors = {}

    for cent2change in range(kmeans.num_centroids):
        
        ## to separate the cost function within the different assignments
        assigned_regions[cent2change] = []

        # Get initial centroids
        centroids = copy.deepcopy(kmeans.centroids)

        # to keep loss
        loss_acc[cent2change] = []
        
        for _c in c_coord_range:
            centroids[cent2change,:] = _c

            centroids_assigned, X_assigned = ...

            assigned_regions[cent2change].append(tuple(centroids_assigned))

            loss = ...
            loss_acc[cent2change].append(loss)
     
        unique_assignments = list(set(assigned_regions[cent2change]))
        region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
       
        ## ==================
        ## Draw loss function
        ax_list_1[cent2change].plot(...)
        ax_list_1[cent2change].scatter(...)

        ax_list_1[cent2change].set_xlabel(...)

**Solution:**

In [ ]:
# Plotting specifications
N_grid = 100
grid_lim_l = -10
grid_lim_u = 10
## coordinate range of values to change
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,1, figsize = (10,20))

## Locally convex vs wthing assginment change
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X, centroids = kmeans.centroids)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    # Get initial centroids
    centroids = copy.deepcopy(kmeans.centroids)

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
    
        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)
 
    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change].set_xlabel(f"Centroid {cent2change}")